# DB2Model — QLoRA на Kaggle

Один прогон даёт **оба** числа для сравнения, на одной модели и одних вопросах:

| Арм | Схема в промпте | Адаптер |
|---|---|---|
| `baseline` | да | выключен |
| `lora` | нет | включён |

Это и есть главный вопрос ветки: знание в контексте против знания в весах.
Сравнивать между собой можно только числа из одного прогона — модель, вопросы
и генерация одинаковые, отличается ровно то, что проверяем.

## Перед запуском

1. **Settings → Accelerator → GPU** (T4 x2 или P100).
2. **Settings → Internet → On**, иначе модель не скачается.
3. Залей папку `db2model/kaggle_input/` как Kaggle Dataset и подключи через **Add Input**.
   Внутри должны быть: `train.json`, `val.json`, `bird_large.json` и профили `*_profile.json`.
4. Поправь `DATA_DIR` под путь своего датасета.

## Что забрать домой

- `/kaggle/working/query_results_lora.json`
- `/kaggle/working/query_results_baseline.json`
- `/kaggle/working/adapter/`

EX считается дома: базы из Kaggle не видно, она за твоим ssh-туннелем.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [ ]:
import json, re, torch
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# ПОПРАВЬ ПОД СВОЙ ДАТАСЕТ
DATA_DIR = Path("/kaggle/input/datasets/vorange/db2model-lora")

OUT_DIR = Path("/kaggle/working")
ADAPTER_DIR = OUT_DIR / "adapter"

train_pairs = json.loads((DATA_DIR / "train.json").read_text(encoding="utf-8"))
val_pairs = json.loads((DATA_DIR / "val.json").read_text(encoding="utf-8"))
bird = json.loads((DATA_DIR / "bird_large.json").read_text(encoding="utf-8"))

DBS = sorted({p["db_id"] for p in train_pairs})
profiles = {db: json.loads((DATA_DIR / f"{db}_profile.json").read_text(encoding="utf-8"))
            for db in DBS}

print("GPU:", torch.cuda.get_device_name(0))
print("памяти всего:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "ГБ")
print("базы:", DBS)
print("train:", len(train_pairs), "| val:", len(val_pairs))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers

print("transformers", transformers.__version__)

# Тип весов обязан совпадать с флагом тренера: bf16-веса плюс fp16=True роняют
# GradScaler. На T4 is_bf16_supported() возвращает True через эмуляцию — работает,
# но медленнее; поставь float16 вручную, если упрёшься во время.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("считаем в", COMPUTE_DTYPE)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# transformers v5 переименовал torch_dtype в dtype; под старым именем аргумент
# игнорируется молча и модель грузится в своём родном bfloat16.
load_kwargs = dict(quantization_config=bnb, device_map={"": 0})
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **load_kwargs)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **load_kwargs)
model.config.use_cache = False

dtypes = {p.dtype for p in model.parameters()}
print("типы параметров:", dtypes)
assert COMPUTE_DTYPE in dtypes, f"веса не в {COMPUTE_DTYPE} — обучение упадёт на GradScaler"
print("занято на GPU:", round(torch.cuda.memory_allocated() / 1e9, 2), "ГБ")

## Промпты двух армов

Отличие ровно одно — есть схема в промпте или нет. Всё остальное совпадает,
иначе сравнение нечестное.

In [ ]:
def schema_text(db: str) -> str:
    """Лёгкая схема, как её подаёт стенд в baseline: таблицы, колонки, типы."""
    lines = []
    for table, info in profiles[db]["tables"].items():
        cols = ", ".join(f"{c['name']} {c['type']}" for c in info["columns"])
        lines.append(f"{table}({cols})")
    return "\n".join(lines)


def build_prompt(db: str, question: str, with_schema: bool) -> str:
    system = f"You are a PostgreSQL expert for the database `{db}`. Return only SQL."
    if with_schema:
        system += f"\n\nSchema:\n{schema_text(db)}"
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True,
    )


def generate(gen_model, prompt: str, max_new_tokens: int = 200) -> tuple[str, int]:
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, inputs["input_ids"].shape[1]


def clean_sql(text: str) -> str:
    text = re.sub(r"^```(?:sql)?|```$", "", text.strip(), flags=re.IGNORECASE)
    return text.strip().rstrip(";").strip()


for db in DBS:
    with_len = len(tokenizer(build_prompt(db, "q", True))["input_ids"])
    without_len = len(tokenizer(build_prompt(db, "q", False))["input_ids"])
    print(f"{db:<22} со схемой {with_len:>5} ток. | без схемы {without_len:>4} ток.")

In [ ]:
from datasets import Dataset

def to_text(p):
    # Обучаем без схемы: знание о базе должно осесть в весах.
    return {"text": build_prompt(p["db_id"], p["question"], with_schema=False)
                    + p["sql"] + tokenizer.eos_token}

train_ds = Dataset.from_list([to_text(p) for p in train_pairs])
val_ds = Dataset.from_list([to_text(p) for p in val_pairs])
print(train_ds, val_ds, sep="\n")
print("\nпример:\n", train_ds[0]["text"][:400])

In [ ]:
from dataclasses import fields

import trl
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("trl", trl.__version__)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

use_bf16 = COMPUTE_DTYPE is torch.bfloat16
kwargs = dict(
    output_dir=str(OUT_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    report_to="none",
)

# trl переименовал max_seq_length в max_length; спрашиваем у класса, а не гадаем.
supported = {f.name for f in fields(SFTConfig)}
for name in ("max_length", "max_seq_length"):
    if name in supported:
        kwargs[name] = 512
        break

dropped = set(kwargs) - supported
if dropped:
    print("эта версия trl не знает и я их не передаю:", dropped)
args = SFTConfig(**{k: v for k, v in kwargs.items() if k in supported})

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=val_ds, peft_config=peft_config)
trainer.train()
print("пик памяти:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "ГБ")

In [ ]:
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("размер адаптера, МБ:",
      round(sum(f.stat().st_size for f in ADAPTER_DIR.iterdir()) / 1e6, 1))

## Оба арма

`disable_adapter()` возвращает исходную модель — то есть baseline считается той же
3B, теми же вопросами, тем же кодом. Отличается только схема в промпте и адаптер.

In [ ]:
import contextlib

model.config.use_cache = True
trainer.model.eval()

questions = [q for q in bird if q["db_id"] in DBS]
print("вопросов:", len(questions), "по базам:", DBS)


def run_arm(name: str, with_schema: bool, use_adapter: bool) -> dict:
    predictions, prompt_tokens = {}, []
    ctx = contextlib.nullcontext() if use_adapter else trainer.model.disable_adapter()
    with ctx:
        for i, q in enumerate(questions, 1):
            question = f"question: {q['question']}, evidence (may be empty): {q['evidence']}"
            prompt = build_prompt(q["db_id"], question, with_schema)
            text, n_tok = generate(trainer.model, prompt)
            predictions[str(q["question_id"])] = clean_sql(text)
            prompt_tokens.append(n_tok)
            if i % 20 == 0:
                print(f"  {name}: {i}/{len(questions)}")

    out = OUT_DIR / f"query_results_{name}.json"
    out.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    avg = sum(prompt_tokens) / len(prompt_tokens)
    print(f"{name}: {out}, prompt-токенов на вопрос в среднем {avg:.0f}")
    return {"file": str(out), "avg_prompt_tokens": avg}


stats = {
    "baseline": run_arm("baseline", with_schema=True, use_adapter=False),
    "lora": run_arm("lora", with_schema=False, use_adapter=True),
}
(OUT_DIR / "token_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
print("\n", json.dumps(stats, indent=2, ensure_ascii=False))

## Дома

Скачай оба файла предсказаний и посчитай EX через туннель:

```bash
PYTHONIOENCODING=utf-8 uv run --env-file .env python bird_evaluate_only.py \
    query_results_baseline.json data/bird_large.json
PYTHONIOENCODING=utf-8 uv run --env-file .env python bird_evaluate_only.py \
    query_results_lora.json data/bird_large.json
```

Вот эти два числа плюс `token_stats.json` и есть результат ветки: точность против
контекста. Если EX у `lora` ниже — так и пишем, это честный результат.